# Clase 5 — Funciones Hash Criptográficas

> **Curso:** Criptografía Aplicada  
> **Objetivo:** Comprender el diseño, propiedades de seguridad e implementación práctica de las familias SHA-2, SHA-3/Keccak y BLAKE2/BLAKE3. Estudiar las estructuras de Merkle Trees y los principales ataques: *birthday attack* y *length-extension attack*.

---

## Tabla de Contenidos

1. [¿Qué es una función hash criptográfica?](#1)
2. [Propiedades de seguridad](#2)
   - 2.1 [Resistencia a preimagen](#2-1)
   - 2.2 [Resistencia a segunda preimagen](#2-2)
   - 2.3 [Resistencia a colisiones](#2-3)
   - 2.4 [Relaciones entre propiedades](#2-4)
3. [SHA-2](#3)
   - 3.1 [Construcción Merkle-Damgård](#3-1)
   - 3.2 [Función de compresión SHA-256](#3-2)
   - 3.3 [Implementación SHA-256 desde cero](#3-3)
   - 3.4 [SHA-512 y variantes truncadas](#3-4)
   - 3.5 [Uso con `hashlib`](#3-5)
4. [SHA-3 / Keccak](#4)
   - 4.1 [Construcción esponja (sponge)](#4-1)
   - 4.2 [Permutación Keccak-f[1600]](#4-2)
   - 4.3 [SHAKE: XOFs extensibles](#4-3)
   - 4.4 [Práctica con `hashlib`](#4-4)
5. [BLAKE2 y BLAKE3](#5)
   - 5.1 [BLAKE2b / BLAKE2s](#5-1)
   - 5.2 [BLAKE3 — árbol interno y paralelismo](#5-2)
   - 5.3 [Keyed hashing y derivación de claves](#5-3)
6. [Merkle Trees](#6)
   - 6.1 [Construcción e implementación](#6-1)
   - 6.2 [Pruebas de inclusión (Merkle Proof)](#6-2)
   - 6.3 [Aplicaciones: Bitcoin, TLS, Git](#6-3)
7. [Birthday Attack](#7)
   - 7.1 [Paradoja del cumpleaños](#7-1)
   - 7.2 [Simulación y curva teórica](#7-2)
   - 7.3 [Implicaciones en tamaño de digest](#7-3)
8. [Length-Extension Attack](#8)
   - 8.1 [Vulnerabilidad en Merkle-Damgård](#8-1)
   - 8.2 [Demo: forjando un MAC con SHA-256](#8-2)
   - 8.3 [Mitigaciones (HMAC, SHA-3, BLAKE)](#8-3)
9. [Comparativa de rendimiento](#9)
10. [Ejercicios propuestos](#10)
11. [Referencias](#11)

---


## Importaciones globales

In [ ]:
import hashlib, hmac, os, struct, time, math, random, secrets
from itertools import accumulate

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

print('Entorno listo ✓')
print('Python hashlib algorithms:', sorted(hashlib.algorithms_guaranteed))


---
<a id='1'></a>
## 1. ¿Qué es una función hash criptográfica?

Una **función hash criptográfica** es una función determinista:

$$H : \{0,1\}^* \;\ \longrightarrow \;\ \{0,1\}^n$$

que comprime una entrada de longitud arbitraria en un *digest* de tamaño fijo `n` bits.

### Diferencias respecto de hashes no criptográficos

| Característica | Hash no criptográfico (CRC32, FNV, MurmurHash) | Hash criptográfico (SHA-256, BLAKE3) |
|---|---|---|
| Objetivo | Integridad trivial, tablas hash | Integridad criptográfica, autenticación |
| Resistencia a ataques | No | Sí (preimagen, colisión) |
| Velocidad | Muy alta | Alta (pero diseñada con seguridad) |
| Tamaño digest | 32–128 bits | 128–512 bits |
| Llave secreta | No | Opcional (keyed hash) |

### Usos principales

```
Funciones hash criptográficas
├── Verificación de integridad de archivos
├── Almacenamiento seguro de contraseñas (bcrypt, Argon2)
├── Firmas digitales  (H(m) se firma, no m)
├── MACs → HMAC, BLAKE2 keyed
├── Derivación de claves (HKDF, KDF)
├── Merkle Trees → blockchains, TLS, Git
├── Compromisos criptográficos (commitment schemes)
└── Generación pseudoaleatoria (PRNG)
```

> 🎯 **Idea clave:** el digest actúa como *huella digital* del mensaje; cualquier cambio en la entrada produce un digest completamente diferente.


In [ ]:
# Demostración de efecto avalancha — un bit cambia todo el digest
msg1 = b'La criptografía es fascinante.'
msg2 = b'La criptografía es fasciname.'   # falta una 't'

h1 = hashlib.sha256(msg1).digest()
h2 = hashlib.sha256(msg2).digest()

bits_diferentes = bin(int.from_bytes(bytes(a ^ b for a, b in zip(h1, h2)), 'big')).count('1')

print(f'SHA-256 msg1 : {h1.hex()}')
print(f'SHA-256 msg2 : {h2.hex()}')
print(f'Bits distintos: {bits_diferentes} / 256  ({100*bits_diferentes/256:.1f}%)')

# Visualizar diferencia bit a bit
diff_bits = [int(b) for byte in bytes(a ^ b for a, b in zip(h1, h2)) for b in f'{byte:08b}']
fig, ax = plt.subplots(figsize=(12, 2))
ax.imshow([diff_bits], cmap='RdYlGn_r', aspect='auto')
ax.set_title('Bits distintos entre SHA-256(msg1) y SHA-256(msg2) (rojo = distinto)')
ax.set_xlabel('Posición del bit (0–255)')
ax.set_yticks([])
plt.tight_layout()
plt.show()


---
<a id='2'></a>
## 2. Propiedades de seguridad

Las tres propiedades fundamentales se definen formalmente para un hash `H` con digest de `n` bits.

<a id='2-1'></a>
### 2.1 Resistencia a preimagen (*Preimage Resistance*)

> Dado un digest `y`, es computacionalmente infactible encontrar **cualquier** `x` tal que `H(x) = y`.

- Seguridad esperada: **O(2ⁿ)** operaciones.
- Garantiza que no se puede *revertir* el hash.
- Crítica para: almacenamiento de contraseñas, compromisos.

```
Atacante conoce:  y = H(x)
Meta:             encontrar x' tal que H(x') = y
Costo bruto:      2^n evaluaciones de H
```

<a id='2-2'></a>
### 2.2 Resistencia a segunda preimagen (*Second Preimage Resistance*)

> Dado un mensaje `x`, es infactible encontrar un `x' ≠ x` tal que `H(x') = H(x)`.

- Seguridad esperada: **O(2ⁿ)** operaciones.
- Diferencia clave con colisión: el atacante *tiene fijo* `x`.
- Crítica para: integridad de firmware, documentos firmados.

<a id='2-3'></a>
### 2.3 Resistencia a colisiones (*Collision Resistance*)

> Es infactible encontrar **cualquier par** `(x, x')` con `x ≠ x'` tal que `H(x) = H(x')`.

- Seguridad esperada: **O(2^(n/2))** operaciones (birthday bound).
- Propiedad más fuerte que exige: colisión ≠ segunda preimagen.
- Crítica para: firmas digitales, Merkle Trees.

> ⚠️ Por el argumento del cumpleaños, la resistencia a colisiones es **siempre más débil** que la resistencia a preimagen.

<a id='2-4'></a>
### 2.4 Relaciones entre propiedades

```
Resistencia a colisiones
        ↓  (implica)
Resistencia a segunda preimagen
        ↓  (implica bajo condiciones adicionales)
Resistencia a preimagen
```

| Propiedad | Seguridad teórica | Nivel recomendado hoy |
|---|---|---|
| Preimagen | 2ⁿ | n ≥ 128 bits |
| Segunda preimagen | 2ⁿ | n ≥ 128 bits |
| Colisión | 2^(n/2) | n ≥ 256 bits |

### Resumen visual


In [ ]:
# Diagrama comparativo de seguridad por tamaño de digest
sizes = [128, 160, 224, 256, 384, 512]
preimage  = sizes
collision = [n // 2 for n in sizes]
algorithms_map = {
    128: 'MD5 (roto)',
    160: 'SHA-1 (roto)',
    224: 'SHA-224',
    256: 'SHA-256 / SHA3-256',
    384: 'SHA-384 / SHA3-384',
    512: 'SHA-512 / SHA3-512',
}

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(sizes))
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], preimage,  width, label='Resistencia preimagen (bits)', color='steelblue')
bars2 = ax.bar([i + width/2 for i in x], collision, width, label='Resistencia colisión (bits)',   color='tomato')

ax.axhline(128, color='green',  linestyle='--', linewidth=1.2, label='Umbral seguro: 128 bits')
ax.axhline(80,  color='orange', linestyle=':',  linewidth=1.2, label='Umbral débil:   80 bits')
ax.set_xticks(list(x))
ax.set_xticklabels([f'{s}b\n({algorithms_map[s]})' for s in sizes], fontsize=8)
ax.set_ylabel('Bits de seguridad')
ax.set_title('Seguridad por tamaño de digest')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


---
<a id='3'></a>
## 3. SHA-2

La familia **SHA-2** (*Secure Hash Algorithm 2*) fue publicada por NIST en 2001. Incluye:

| Variante | Digest (bits) | Bloque (bits) | Palabra | Rondas |
|---|---|---|---|---|
| SHA-224 | 224 | 512 | 32 | 64 |
| SHA-256 | 256 | 512 | 32 | 64 |
| SHA-384 | 384 | 1024 | 64 | 80 |
| SHA-512 | 512 | 1024 | 64 | 80 |
| SHA-512/224 | 224 | 1024 | 64 | 80 |
| SHA-512/256 | 256 | 1024 | 64 | 80 |

<a id='3-1'></a>
### 3.1 Construcción Merkle-Damgård

SHA-256 sigue la construcción **Merkle-Damgård (MD)**:

```
m₁  m₂  m₃  ... mₖ  (padding + longitud)
│   │   │        │
▼   ▼   ▼        ▼
H₀→[f]→[f]→[f]→...→[f]→ digest
       IV
```

1. **Padding:** se agrega un bit `1`, luego ceros, y finalmente la longitud del mensaje en 64 bits, para que el total sea múltiplo de 512 bits.
2. **IV:** valores iniciales derivados de raíces cuadradas de los primeros primos.
3. **Función de compresión `f`:** toma el estado de 256 bits y el bloque de 512 bits; produce un nuevo estado de 256 bits.

> ⚠️ **Limitación MD:** la construcción es vulnerable al *length-extension attack* (ver sección 8).

<a id='3-2'></a>
### 3.2 Función de compresión SHA-256

En cada ronda `i` (0–63):

```
T1 = h + Σ1(e) + Ch(e,f,g) + K[i] + W[i]
T2 = Σ0(a) + Maj(a,b,c)
h = g;  g = f;  f = e;  e = d + T1
d = c;  c = b;  b = a;  a = T1 + T2
```

Donde:
- `Σ0(a) = ROTR²(a) ⊕ ROTR¹³(a) ⊕ ROTR²²(a)`
- `Σ1(e) = ROTR⁶(e) ⊕ ROTR¹¹(e) ⊕ ROTR²⁵(e)`
- `Ch(e,f,g) = (e AND f) ⊕ (¬e AND g)`
- `Maj(a,b,c) = (a AND b) ⊕ (a AND c) ⊕ (b AND c)`
- `W[i]` = schedule del mensaje (expansión a 64 palabras de 32 bits)
- `K[i]` = constantes derivadas de ∛ de los primeros 64 primos


<a id='3-3'></a>
### 3.3 Implementación SHA-256 desde cero

Implementamos SHA-256 completo en Python puro para comprender cada paso interno. **No usar en producción — usar `hashlib`.**


In [ ]:
# SHA-256 implementación educativa completa
# Referencia: FIPS PUB 180-4

class SHA256:
    """SHA-256 implementado desde cero (solo educativo)."""

    # Primeras 32 bits de las raíces cuadradas de los primeros 8 primos
    H0 = [
        0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
        0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19,
    ]

    # Primeras 32 bits de las raíces cúbicas de los primeros 64 primos
    K = [
        0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
        0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
        0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
        0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
        0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
        0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
        0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
        0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
        0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
        0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
        0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
        0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
        0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
        0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
        0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
        0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
    ]

    MASK = 0xFFFFFFFF

    @staticmethod
    def rotr(x: int, n: int) -> int:
        return ((x >> n) | (x << (32 - n))) & SHA256.MASK

    @staticmethod
    def _pad(msg: bytes) -> bytes:
        """Padding Merkle-Damgård: agrega 0x80, ceros y longitud en 64 bits big-endian."""
        length = len(msg) * 8       # longitud en bits
        msg += b'\x80'
        msg += b'\x00' * ((55 - len(msg)) % 64)  # rellena hasta 56 mod 64
        msg += struct.pack('>Q', length)           # longitud en 64 bits BE
        return msg

    @staticmethod
    def _schedule(block: bytes) -> list:
        """Expande el bloque de 512 bits en 64 palabras de 32 bits."""
        W = list(struct.unpack('>16I', block))
        for i in range(16, 64):
            s0 = SHA256.rotr(W[i-15], 7) ^ SHA256.rotr(W[i-15], 18) ^ (W[i-15] >> 3)
            s1 = SHA256.rotr(W[i-2],  17) ^ SHA256.rotr(W[i-2],  19) ^ (W[i-2]  >> 10)
            W.append((W[i-16] + s0 + W[i-7] + s1) & SHA256.MASK)
        return W

    @staticmethod
    def _compress(state: list, block: bytes) -> list:
        """Función de compresión: procesa un bloque de 512 bits."""
        W = SHA256._schedule(block)
        a, b, c, d, e, f, g, h = state
        K, MASK, rotr = SHA256.K, SHA256.MASK, SHA256.rotr

        for i in range(64):
            S1  = rotr(e, 6) ^ rotr(e, 11) ^ rotr(e, 25)
            ch  = (e & f) ^ (~e & g)
            T1  = (h + S1 + ch + K[i] + W[i]) & MASK
            S0  = rotr(a, 2) ^ rotr(a, 13) ^ rotr(a, 22)
            maj = (a & b) ^ (a & c) ^ (b & c)
            T2  = (S0 + maj) & MASK

            h = g; g = f; f = e
            e = (d + T1) & MASK
            d = c; c = b; b = a
            a = (T1 + T2) & MASK

        return [(x + y) & MASK for x, y in zip(state, [a,b,c,d,e,f,g,h])]

    @classmethod
    def hash(cls, msg: bytes) -> bytes:
        """Calcula SHA-256 de msg y devuelve 32 bytes."""
        padded = cls._pad(msg)
        state  = cls.H0[:]
        for i in range(0, len(padded), 64):
            state = cls._compress(state, padded[i:i+64])
        return struct.pack('>8I', *state)


# Verificación contra hashlib
test_vectors = [
    b'',
    b'abc',
    b'abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq',
    b'The quick brown fox jumps over the lazy dog',
]

print('Verificación SHA-256 vs hashlib:')
all_ok = True
for msg in test_vectors:
    ours = SHA256.hash(msg).hex()
    ref  = hashlib.sha256(msg).hexdigest()
    ok   = '✓' if ours == ref else '✗'
    if ours != ref:
        all_ok = False
    label = (msg[:40].decode() + '...') if len(msg) > 40 else msg.decode() or '<vacío>'
    print(f'  {ok}  {label!r}')
    if ours != ref:
        print(f'     Nuestro : {ours}')
        print(f'     hashlib : {ref}')

print(f'\nTodos los vectores OK: {all_ok}')


In [ ]:
# Visualización del schedule de mensaje SHA-256
import struct

msg = b'Hola SHA-256 mundo criptográfico'
padded = SHA256._pad(msg)
W = SHA256._schedule(padded[:64])   # primer bloque

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(64), W, color=['steelblue']*16 + ['tomato']*48, alpha=0.8)
ax.set_title('Message Schedule SHA-256: primeras 16 palabras (azul) y 48 expandidas (rojo)')
ax.set_xlabel('Índice W[i]')
ax.set_ylabel('Valor (hex)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):#010x}'))
plt.tight_layout()
plt.show()


<a id='3-4'></a>
### 3.4 SHA-512 y variantes truncadas

SHA-512 opera con **palabras de 64 bits** y bloques de **1024 bits**. Las constantes y rotaciones cambian:

| Parámetro | SHA-256 | SHA-512 |
|---|---|---|
| Tamaño de palabra | 32 bits | 64 bits |
| Tamaño de bloque | 512 bits | 1024 bits |
| Rondas | 64 | 80 |
| IV | ∛ primeros 8 primos | ∛ primeros 8 primos |
| Rotaciones Σ0 | (2,13,22) | (28,34,39) |
| Rotaciones Σ1 | (6,11,25) | (14,18,41) |
| Constantes K | 64 × 32 bits | 80 × 64 bits |

**SHA-512/256**: ejecuta SHA-512 con IV diferente (calculado con `ΣSHA-512/t IV Gen`) y trunca a 256 bits. Más rápido que SHA-256 en CPUs de 64 bits.


In [ ]:
# SHA-2 con hashlib — todas las variantes
msg = b'Criptografia SHA-2'

variantes = ['sha224', 'sha256', 'sha384', 'sha512', 'sha512_224', 'sha512_256']
print(f'Mensaje: {msg.decode()!r}\n')
print(f'{"Algoritmo":<15} {"Digest (hex)":<70} {"Bytes"}')
print('-' * 95)
for v in variantes:
    h = hashlib.new(v, msg).hexdigest()
    print(f'{v:<15} {h:<70} {len(h)//2}')


<a id='3-5'></a>
### 3.5 Uso con `hashlib` (streaming y archivos)


In [ ]:
import io

def sha256_file(path_or_bytes) -> str:
    """Calcula SHA-256 de un archivo o buffer en modo streaming."""
    h = hashlib.sha256()
    if isinstance(path_or_bytes, (bytes, bytearray)):
        buf = io.BytesIO(path_or_bytes)
    else:
        buf = open(path_or_bytes, 'rb')
    with buf:
        while chunk := buf.read(65536):
            h.update(chunk)
    return h.hexdigest()


# Nota: si `path_or_bytes` es una ruta de archivo, un FileNotFoundError se propaga
# intencionalmente para que el código llamador lo gestione.

# Simular archivo grande
data = secrets.token_bytes(1_000_000)   # 1 MB
digest = sha256_file(data)
print(f'SHA-256 de 1 MB de datos aleatorios: {digest}')

# API incremental (útil para protocolos)
ctx = hashlib.sha256()
ctx.update(b'parte1 ')
ctx.update(b'parte2 ')
ctx.update(b'parte3')
print(f'Hash incremental : {ctx.hexdigest()}')
print(f'Hash de una vez  : {hashlib.sha256(b"parte1 parte2 parte3").hexdigest()}')
print(f'Equivalentes     : {ctx.hexdigest() == hashlib.sha256(b"parte1 parte2 parte3").hexdigest()}')

# copy() del estado intermedio
ctx2 = ctx.copy()
ctx2.update(b' extra')
print(f'\nCopia + extra    : {ctx2.hexdigest()}')
print(f'Original         : {ctx.hexdigest()}')


---
<a id='4'></a>
## 4. SHA-3 / Keccak

SHA-3 fue seleccionado en 2012 mediante competencia NIST (2007–2012). Su algoritmo base es **Keccak**, diseñado por Bertoni, Daemen, Peeters y Van Assche.

**Motivación:** SHA-2 es seguro, pero comparte estructura MD con MD5/SHA-1. SHA-3 ofrece un diseño completamente diferente como respaldo.

<a id='4-1'></a>
### 4.1 Construcción esponja (*Sponge Construction*)

```
       Absorción                    Exprimido
┌────────────────────────┐    ┌───────────────────┐
│  m₁   m₂   m₃   mₖ  │    │  z₁   z₂   ...    │
│  ↓    ↓    ↓    ↓   │    │  ↑    ↑            │
│ [⊕]  [⊕]  [⊕]  [⊕]  │    │  │    │            │
│  │    │    │    │   │    │  │    │            │
│ [f]→ [f]→ [f]→ [f]  │ →  │ [f]→ [f]→ ...     │
│  └────────────────┘  │    │                   │
│   rate r   cap c     │    │   rate r   cap c  │
└────────────────────────┘    └───────────────────┘
```

**Estado:** 1600 bits = rate `r` + capacity `c`
- SHA3-256: r=1088, c=512
- SHA3-512: r=576,  c=1024

**Ventajas sobre MD:**
- **Sin length-extension attack** (la capacidad nunca se expone)
- **XOF** (eXtendable Output Function): SHAKE128, SHAKE256
- Diseño más simple de analizar formalmente

<a id='4-2'></a>
### 4.2 Permutación Keccak-f[1600]

El estado de 1600 bits se organiza en una matriz 5×5 de carriles de 64 bits. Cada ronda aplica 5 pasos:

| Paso | Operación | Propósito |
|---|---|---|
| **θ (theta)** | Paridad de columnas | Difusión lineal |
| **ρ (rho)**   | Rotaciones de carriles | Dispersión de bits |
| **π (pi)**    | Permutación de posiciones | Difusión entre carriles |
| **χ (chi)**   | Sustitución no lineal (único no lineal) | Confusión |
| **ι (iota)**  | XOR con constante de ronda | Rompe simetría |

Keccak-f[1600] realiza **24 rondas**.


In [ ]:
# Implementación simplificada de la función θ (theta) de Keccak para ilustración
def keccak_theta(A):
    """Paso theta de Keccak. A: lista 5x5 de enteros de 64 bits."""
    MASK64 = (1 << 64) - 1
    C = [A[x][0] ^ A[x][1] ^ A[x][2] ^ A[x][3] ^ A[x][4] for x in range(5)]
    D = [C[(x-1) % 5] ^ (((C[(x+1) % 5] << 1) | (C[(x+1) % 5] >> 63)) & MASK64)
         for x in range(5)]
    return [[A[x][y] ^ D[x] for y in range(5)] for x in range(5)]

# Estado de ejemplo (5x5 de 64 bits)
import random
random.seed(42)
A = [[random.getrandbits(64) for _ in range(5)] for _ in range(5)]
A_theta = keccak_theta(A)

print('Estado original A[0][0]:', hex(A[0][0]))
print('Tras θ:        A[0][0]:', hex(A_theta[0][0]))
print('Diferencia XOR:        ', hex(A[0][0] ^ A_theta[0][0]))


In [ ]:
# Visualización de la difusión en el estado Keccak 5x5
import numpy as np

def bits_matrix(A):
    """Convierte matriz 5x5 de uint64 en matriz 5x64 de bits."""
    return np.array([[int(b) for b in f'{A[x][y]:064b}'] for x in range(5) for y in range(5)])

M_original = bits_matrix(A)
M_theta    = bits_matrix(A_theta)
M_diff     = M_original ^ M_theta

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, mat, title in zip(axes,
                           [M_original, M_theta, M_diff],
                           ['Estado original', 'Tras θ (theta)', 'Diferencia (XOR)']):
    ax.imshow(mat, cmap='Blues', aspect='auto', interpolation='none')
    ax.set_title(title)
    ax.set_xlabel('Bit dentro del carril (0-63)')
    ax.set_ylabel('Carril (x·5+y)')
plt.suptitle('Efecto del paso θ en el estado Keccak 5×5×64 bits', y=1.01)
plt.tight_layout()
plt.show()
print(f'Bits modificados por θ: {M_diff.sum()} / {5*5*64} ({100*M_diff.sum()/(5*5*64):.1f}%)')


<a id='4-3'></a>
### 4.3 SHAKE: XOFs extensibles

SHAKE128 y SHAKE256 son **eXtendable Output Functions**: producen un digest de longitud arbitraria.

```python
hashlib.shake_256(msg).digest(length=64)   # 64 bytes
hashlib.shake_256(msg).digest(length=128)  # 128 bytes — mismo estado, más salida
```

Aplicaciones: KDF, generación de matrices en algoritmos post-cuánticos (CRYSTALS-Kyber usa SHAKE).

<a id='4-4'></a>
### 4.4 Práctica con `hashlib`


In [ ]:
msg = b'Keccak y SHA-3'

# SHA-3 fijo
for alg in ['sha3_224', 'sha3_256', 'sha3_384', 'sha3_512']:
    h = hashlib.new(alg, msg).hexdigest()
    print(f'{alg:<12}: {h}')

print()
# SHAKE — output de longitud variable
for length in [16, 32, 64, 128]:
    h = hashlib.shake_256(msg).digest(length)
    print(f'SHAKE256({length:3d} bytes): {h.hex()}')

print()
# Comparación SHA-256 vs SHA3-256 (difieren a pesar del mismo tamaño)
h_sha2  = hashlib.sha256(msg).hexdigest()
h_sha3  = hashlib.sha3_256(msg).hexdigest()
bits_dif = bin(int(h_sha2, 16) ^ int(h_sha3, 16)).count('1')
print(f'SHA-256   : {h_sha2}')
print(f'SHA3-256  : {h_sha3}')
print(f'Bits distintos entre SHA-256 y SHA3-256 del mismo mensaje: {bits_dif}')


---
<a id='5'></a>
## 5. BLAKE2 y BLAKE3

### Historia

```
HASH256 (2004) → BLAKE (2008, finalista SHA-3) → BLAKE2 (2012) → BLAKE3 (2020)
```

<a id='5-1'></a>
### 5.1 BLAKE2b / BLAKE2s

Publicado en 2012. Más rápido que SHA-256 y SHA-3 en software.

| Variante | Arquitectura | Digest máx. | Bloque | Palabras |
|---|---|---|---|---|
| BLAKE2b | 64-bit | 512 bits | 1024 bits | 8×64 |
| BLAKE2s | 32-bit (ARM) | 256 bits | 512 bits | 8×32 |

**Características diferenciales:**
- Keyed hashing nativo (sin HMAC externo)
- Derivación de claves integrada
- Hash personalizado (*personalization string*)
- Árbol de hash para paralelismo
- Basado en ChaCha (función G similar a quarterround)

<a id='5-2'></a>
### 5.2 BLAKE3 — árbol interno y paralelismo

BLAKE3 (2020) es la versión más moderna, con paralelismo automático usando **árbol Merkle interno**:

```
                  Raíz
              ┌────┴────┐
           Nodo1     Nodo2
          ┌──┴──┐   ┌──┴──┐
        C1  C2  C3  C4  (chunks de 1 KB)
```

- Cada chunk se procesa con 7 rondas de la función de compresión
- Los chunks se combinan binariamente en árbol
- ≥ 10× más rápido que SHA-256 en hardware moderno
- XOF: salida de longitud arbitraria como SHAKE

<a id='5-3'></a>
### 5.3 Práctica con `hashlib` y `blake3`


In [ ]:
msg  = b'BLAKE hashing is fast and secure'
key  = b'mi_clave_secreta_32bytes_exactos!'

# BLAKE2b — digest variable (1–64 bytes)
b2b_32 = hashlib.blake2b(msg, digest_size=32).hexdigest()
b2b_64 = hashlib.blake2b(msg, digest_size=64).hexdigest()
# BLAKE2s — digest variable (1–32 bytes)
b2s_32 = hashlib.blake2s(msg, digest_size=32).hexdigest()

print('BLAKE2b (32B):', b2b_32)
print('BLAKE2b (64B):', b2b_64)
print('BLAKE2s (32B):', b2s_32)

print()
# Keyed hashing — equivalente funcional a HMAC
keyed_2b = hashlib.blake2b(msg, key=key[:32], digest_size=32).hexdigest()
keyed_2s = hashlib.blake2s(msg, key=key[:32], digest_size=32).hexdigest()
print('BLAKE2b keyed:', keyed_2b)
print('BLAKE2s keyed:', keyed_2s)

print()
# Personalización
h_default    = hashlib.blake2b(msg, digest_size=32).hexdigest()
h_custom_ctx = hashlib.blake2b(msg, digest_size=32, person=b'MiApp_v2').hexdigest()
print('Sin personalización  :', h_default)
print('Con personalización  :', h_custom_ctx)
print('Son distintos        :', h_default != h_custom_ctx)


In [ ]:
# Intentar usar blake3 si está instalado, si no, mostrar comparativa con BLAKE2b
try:
    import blake3 as _blake3
    b3 = _blake3.blake3(msg).hexdigest()
    b3_keyed = _blake3.blake3(msg, key=key[:32]).hexdigest()
    print('BLAKE3         :', b3)
    print('BLAKE3 keyed   :', b3_keyed)
    # XOF
    xof_64 = _blake3.blake3(msg).digest(64)
    print('BLAKE3 XOF 64B :', xof_64.hex())
except (ImportError, AttributeError):
    print('blake3 no instalado — instalar con: pip install blake3')
    print('Usando BLAKE2b como referencia comparativa:')
    for digest_size in [32, 48, 64]:
        h = hashlib.blake2b(msg, digest_size=digest_size).hexdigest()
        print(f'  BLAKE2b({digest_size}B): {h}')


---
<a id='6'></a>
## 6. Merkle Trees

Un **árbol de Merkle** (*Merkle Hash Tree*) es una estructura de árbol binario donde:
- Las **hojas** contienen hashes de los datos.
- Los **nodos internos** contienen el hash de la concatenación de sus hijos.
- La **raíz** (*Merkle root*) resume todo el conjunto de datos.

```
                Root = H(N12 ‖ N34)
               /                    \
        N12 = H(H1 ‖ H2)     N34 = H(H3 ‖ H4)
           /        \             /        \
        H1=H(d1)  H2=H(d2)  H3=H(d3)  H4=H(d4)
          |          |         |          |
          d1         d2        d3         d4
```

### Aplicaciones

| Sistema | Uso |
|---|---|
| **Bitcoin** | Merkle root en el encabezado de cada bloque; prueba eficiente de transacciones |
| **Git** | Árbol de objetos (commit → tree → blobs) |
| **TLS/CT** | Certificate Transparency: logs auditables |
| **IPFS** | Direccionamiento por contenido (CID) |
| **ZFS/Btrfs** | Verificación de integridad de bloques de disco |
| **Ethereum** | Patricia-Merkle Trie para estado del mundo |

<a id='6-1'></a>
### 6.1 Construcción e implementación


In [ ]:
import hashlib
from typing import List, Optional, Tuple


def merkle_hash(data: bytes) -> bytes:
    """Nodo hoja: doble hash SHA-256 (convención Bitcoin / SHA256d).
    El doble hash es específico de Bitcoin para mitigar ataques de extensión
    de longitud. Otros sistemas (ej. Certificate Transparency, Git) usan
    SHA-256 simple. Adaptar según el protocolo objetivo.
    """
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()


def merkle_parent(left: bytes, right: bytes) -> bytes:
    """Nodo interno: hash de concatenación de hijos."""
    return hashlib.sha256(left + right).digest()


class MerkleTree:
    """Árbol de Merkle sobre SHA-256."""

    def __init__(self, leaves: List[bytes]):
        if not leaves:
            raise ValueError('El árbol requiere al menos una hoja')
        self.leaves = [merkle_hash(d) for d in leaves]
        self.layers = self._build(self.leaves)

    @staticmethod
    def _build(nodes: List[bytes]) -> List[List[bytes]]:
        layers = [nodes]
        current = nodes
        while len(current) > 1:
            if len(current) % 2 == 1:
                current = current + [current[-1]]  # duplicar hoja impar (convención)
            current = [merkle_parent(current[i], current[i+1])
                       for i in range(0, len(current), 2)]
            layers.append(current)
        return layers

    @property
    def root(self) -> bytes:
        return self.layers[-1][0]

    def proof(self, index: int) -> List[Tuple[str, bytes]]:
        """Devuelve la prueba de inclusión para la hoja en `index`."""
        path = []
        for layer in self.layers[:-1]:
            if len(layer) % 2 == 1:
                layer = layer + [layer[-1]]
            sibling_idx = index ^ 1   # XOR con 1 = hermano
            side = 'R' if index % 2 == 0 else 'L'
            path.append((side, layer[sibling_idx]))
            index //= 2
        return path

    @staticmethod
    def verify(leaf_data: bytes, proof: List[Tuple[str, bytes]], root: bytes) -> bool:
        """Verifica que leaf_data pertenece al árbol con la raíz `root`."""
        current = merkle_hash(leaf_data)
        for side, sibling in proof:
            if side == 'R':
                current = merkle_parent(current, sibling)
            else:
                current = merkle_parent(sibling, current)
        return current == root


# Construir árbol con 8 transacciones simuladas
txs = [f'tx_{i:03d}: Alice→Bob ${i*7}'.encode() for i in range(8)]
tree = MerkleTree(txs)

print(f'Hojas    : {len(tree.leaves)}')
print(f'Capas    : {len(tree.layers)}')
print(f'Raíz     : {tree.root.hex()}')

# Prueba de inclusión para tx_003
idx = 3
proof = tree.proof(idx)
print(f'\nPrueba de inclusión para hoja {idx} ({txs[idx].decode()}):')
for step, (side, h) in enumerate(proof):
    print(f'  Paso {step+1}: hermano {side} = {h.hex()[:16]}...')

valid = MerkleTree.verify(txs[idx], proof, tree.root)
print(f'\n¿Prueba válida? {valid}')

# Prueba con dato modificado
tampered = b'tx_003: Alice->Eve $999'
print(f'¿Dato modificado pasa verificación? {MerkleTree.verify(tampered, proof, tree.root)}')


In [ ]:
# Visualización del árbol de Merkle
import matplotlib.pyplot as plt
import matplotlib.patches as FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(-0.5, 8.5)
ax.set_ylim(-0.5, 4)
ax.axis('off')
ax.set_title('Árbol de Merkle — 8 transacciones (SHA-256)', fontsize=13)

colors = {
    0: '#AED6F1',  # hojas
    1: '#A9DFBF',
    2: '#F9E79F',
    3: '#F1948A',  # raíz
}

layer_positions = [
    [i for i in range(8)],          # capa 0: 8 hojas
    [0.5 + 2*i for i in range(4)],  # capa 1: 4 nodos
    [1.5 + 4*i for i in range(2)],  # capa 2: 2 nodos
    [3.5],                           # capa 3: raíz
]

for layer_idx, (layer_nodes, xpositions) in enumerate(zip(tree.layers, layer_positions)):
    for node_idx, (x, h) in enumerate(zip(xpositions, layer_nodes)):
        y = layer_idx
        color = colors[layer_idx]
        rect = plt.Rectangle((x - 0.45, y - 0.35), 0.9, 0.7,
                               linewidth=1.5, edgecolor='gray',
                               facecolor=color, zorder=2)
        ax.add_patch(rect)
        label = h.hex()[:8] + '...' if layer_idx > 0 else f'H(tx{node_idx})'
        ax.text(x, y, label, ha='center', va='center', fontsize=6.5, zorder=3)
        if layer_idx < 3 and node_idx < len(layer_positions[layer_idx + 1]):
            parent_x = layer_positions[layer_idx + 1][node_idx // 2]
            ax.plot([x, parent_x], [y + 0.35, y + 0.65],
                    color='gray', linewidth=1.2, zorder=1)

# Etiquetas de capas
for layer_idx, label in enumerate(['Hojas (tx)', 'Nivel 1', 'Nivel 2', 'Raíz']):
    ax.text(-0.4, layer_idx, label, ha='right', va='center', fontsize=9,
            fontweight='bold', color='dimgray')

plt.tight_layout()
plt.show()


<a id='6-2'></a>
### 6.2 Pruebas de inclusión (Merkle Proof)

Una prueba de inclusión demuestra que un elemento pertenece al árbol con solo **O(log n)** hashes, sin revelar el resto de los datos.

```
Para verificar tx_003 en un árbol de 8 hojas:
  Necesito:  H(tx_002), H(N01), H(N45_67)      ← 3 hashes = log₂(8)
  Proceso:   H(tx_003) → combinar → combinar → comparar con root
  Costo:     3 operaciones hash en lugar de 8
```

Esto permite que un **cliente ligero** (como una billetera Bitcoin SPV) verifique transacciones sin descargar toda la blockchain.


In [ ]:
# Eficiencia: costo de prueba vs tamaño del árbol
n_values = [2**k for k in range(1, 21)]
proof_sizes = [math.log2(n) for n in n_values]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(n_values, proof_sizes, marker='o', markersize=4, color='steelblue', label='Tamaño prueba O(log n)')
ax.plot(n_values, n_values, linestyle='--', color='tomato', alpha=0.7, label='Sin árbol Merkle O(n)')
ax.set_xscale('log', base=2)
ax.set_yscale('log', base=2)
ax.set_xlabel('Número de hojas (n)')
ax.set_ylabel('Hashes necesarios para la prueba')
ax.set_title('Eficiencia de la prueba de inclusión en Merkle Tree')
ax.legend()
ax.grid(True, alpha=0.3)
# Anotar algunos puntos
for n in [1024, 1_048_576]:
    cost = math.log2(n)
    ax.annotate(f'n={n:,}\n{cost:.0f} hashes', xy=(n, cost),
                xytext=(n, cost*3), ha='center', fontsize=8,
                arrowprops=dict(arrowstyle='->', color='gray'))
plt.tight_layout()
plt.show()


<a id='6-3'></a>
### 6.3 Aplicaciones: Bitcoin, TLS y Git

**Bitcoin:**
```
 Block Header
 ┌─────────────────────────────┐
 │ prev_hash │ merkle_root │ ...│
 └─────────────────────────────┘
          ↑
   H(tx_0, tx_1, ..., tx_n)   ← Merkle root de todas las tx del bloque
```

**Git:**
```
commit → tree → blob    (SHA-1 antes, SHA-256 ahora)
   ↑        ↑
H(árbol)   H(archivo)
```

**Certificate Transparency (RFC 9162):**
- Los certificados TLS se publican en logs append-only basados en Merkle Trees.
- Permite detectar certificados fraudulentos con pruebas de inclusión/consistencia.


---
<a id='7'></a>
## 7. Birthday Attack

<a id='7-1'></a>
### 7.1 La paradoja del cumpleaños

**Pregunta clásica:** ¿Cuántas personas necesitas en una habitación para que con probabilidad > 50% dos compartan cumpleaños?

Respuesta intuitiva incorrecta: 183. **Respuesta real: 23.**

**Generalización:** para una función hash con salida de `n` bits, la probabilidad de encontrar una **colisión** con `k` evaluaciones es aproximadamente:

$$P(k, n) \approx 1 - e^{-k^2 / (2 \cdot 2^n)}$$

Para que `P ≈ 0.5`:

$$k \approx \sqrt{2 \cdot 2^n \cdot \ln 2} \approx 1.177 \cdot 2^{n/2}$$

### Implicación para hashes

| Hash | Bits | Colisión en ~... |
|---|---|---|
| MD5 | 128 | 2^64 ≈ 1.8×10¹⁹ |
| SHA-1 | 160 | 2^80 ≈ 1.2×10²⁴ (roto en 2017: SHAttered) |
| SHA-256 | 256 | 2^128 ≈ 3.4×10³⁸ |
| SHA-512 | 512 | 2^256 ≈ 1.2×10⁷⁷ |

> ⚠️ Esta es la razón por la que para integridad de documentos firmados con RSA-2048 se recomienda SHA-256 (128 bits de seguridad frente a colisiones).

<a id='7-2'></a>
### 7.2 Simulación con hash truncado

Simulamos un birthday attack sobre un hash truncado a `b` bits para observar el fenómeno.


In [ ]:
import random, hashlib

def truncated_hash(data: bytes, bits: int) -> int:
    """SHA-256 truncado a `bits` bits."""
    full = int(hashlib.sha256(data).hexdigest(), 16)
    return full >> (256 - bits)


def birthday_attack(bits: int, max_trials: int = 200_000) -> Optional[int]:
    """Busca colisión en hash de `bits` bits. Devuelve nº de intentos o None."""
    seen = {}
    for i in range(max_trials):
        msg = random.getrandbits(64).to_bytes(8, 'big')
        h   = truncated_hash(msg, bits)
        if h in seen:
            return i + 1, seen[h], msg
        seen[h] = msg
    return None


# Se usa random con semilla fija (no secrets) para resultados reproducibles
# en este contexto educativo. En contextos criptográficos reales usar secrets.
random.seed(0xCAFE)
results = []
for b in [12, 14, 16, 18, 20]:
    outcomes = []
    for _ in range(10):
        r = birthday_attack(b, max_trials=500_000)
        if r is not None:
            outcomes.append(r[0])
    if outcomes:
        mean_trials = sum(outcomes) / len(outcomes)
        theoretical  = 1.177 * 2**(b/2)
        results.append((b, mean_trials, theoretical))
        print(f'{b}-bit hash: media {mean_trials:.0f} intentos  (teórico: {theoretical:.0f})')

# Demo de colisión en 16 bits
b = 16
result = birthday_attack(b)
if result:
    n_trials, msg1, msg2 = result
    h1 = truncated_hash(msg1, b)
    h2 = truncated_hash(msg2, b)
    print(f'\nColisión en {b} bits tras {n_trials} intentos:')
    print(f'  msg1 = {msg1.hex()}  → hash = {h1:#06x}')
    print(f'  msg2 = {msg2.hex()}  → hash = {h2:#06x}')
    print(f'  Colisión válida: {h1 == h2}')


In [ ]:
# Curva teórica vs experimental
if results:
    bits_vals    = [r[0] for r in results]
    experimental = [r[1] for r in results]
    theoretical  = [r[2] for r in results]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(bits_vals, theoretical,  'o--', color='steelblue', label='Teórico: 1.177·2^(b/2)')
    ax.plot(bits_vals, experimental, 's-',  color='tomato',   label='Experimental (promedio 10 runs)')
    ax.set_xlabel('Bits del digest')
    ax.set_ylabel('Número de evaluaciones hash')
    ax.set_title('Birthday Attack: intentos hasta colisión')
    ax.set_yscale('log', base=2)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Probabilidad de colisión según nº de muestras (curva de Ramsey)
n_bits = 256
k_range = [2**(i) for i in range(1, 130)]
probs   = [1 - math.exp(-(k**2) / (2 * 2**n_bits)) for k in k_range]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot([math.log2(k) for k in k_range], probs, color='purple')
ax.axhline(0.5, linestyle='--', color='tomato', label='P=0.5 → k ≈ 2^128')
ax.axvline(128, linestyle='--', color='steelblue', alpha=0.7)
ax.set_xlabel('log₂(k) — log del número de muestras')
ax.set_ylabel('Probabilidad de colisión')
ax.set_title('Probabilidad de colisión vs muestras para SHA-256 (n=256 bits)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


<a id='7-3'></a>
### 7.3 Implicaciones en el diseño de sistemas

**Identificadores únicos (UUIDs, tokens):**
```
Problema de colisión para k tokens con n bits:
  P ≈ k² / 2^(n+1)

Para P < 10⁻⁹ con k = 10⁶ tokens:
  n > 2·log₂(10⁶) + 30 ≈ 70 bits mínimo
→ Usar 128 bits (UUID4 o token_bytes(16))
```

**Signaturas digitales:**
- Si SHA-1 colisiona en 2^80, un atacante puede forjar el documento firmado.
- SHAttered (2017): primera colisión SHA-1 práctica con 6500 GPU-años.
- **CA/Browser Forum:** SHA-1 prohibido en certificados TLS desde 2017.

**Tablas rainbow:**
- Precomputan cadenas de reducciones hash para invertir hashes de contraseñas.
- Derrotadas con salt aleatorio: hace que la tabla sea inútil para cada usuario.


In [ ]:
# Demostración de efectividad del salt contra ataques de diccionario
passwords = ['password', '123456', 'qwerty', 'letmein']
diccionario = [hashlib.md5(p.encode()).hexdigest() for p in passwords]
print('Tabla MD5 sin salt (búsqueda inmediata):')
for p, h in zip(passwords, diccionario):
    print(f'  {p:<15} → {h}')

print()
# Con salt único por usuario
print('Hashes MD5 con salt único (cada salt es diferente):')
for p in passwords:
    salt = secrets.token_bytes(16)
    h = hashlib.md5(salt + p.encode()).hexdigest()
    print(f'  {p:<15} salt={salt.hex()[:8]}... → {h}')

print()
print('Nota: en producción usar hashlib.scrypt, bcrypt o Argon2, nunca MD5/SHA-256 sin KDF.')


---
<a id='8'></a>
## 8. Length-Extension Attack

<a id='8-1'></a>
### 8.1 Vulnerabilidad en la construcción Merkle-Damgård

La construcción Merkle-Damgård es vulnerable al **ataque de extensión de longitud** (*length extension attack*):

**Escenario:** un servidor calcula un MAC naïf:
```
MAC = SHA-256(secret ‖ message)
```

**Problema:** el digest de SHA-256 **es el estado interno** al final del procesamiento. Un atacante que conoce `MAC` y `len(secret)` puede:
1. Reconstruir el estado interno desde el MAC.
2. Continuar el hash con un mensaje adicional `extension`.
3. Obtener `SHA-256(secret ‖ message ‖ padding ‖ extension)` **sin conocer `secret`**.

```
Atacante conoce:
  MAC = H(secret ‖ message)     y     len(secret)

Atacante computa:
  state = descomponer MAC en 8 palabras de 32 bits
  new_MAC = H_desde_state(extension)

Servidor acepta:
  H(secret ‖ message ‖ padding ‖ extension) == new_MAC   ✓
```

**Algoritmos vulnerables:** MD5, SHA-1, SHA-256, SHA-512 (todos los MD excepto SHA-512/256 y SHA-384 truncados).

**Algoritmos NO vulnerables:** SHA-3/Keccak, BLAKE2, BLAKE3, HMAC.

<a id='8-2'></a>
### 8.2 Demostración del ataque


In [ ]:
import struct, hashlib

# ---------------------------------------------------------------------------
# Utilidades para SHA-256 sin padding (acceso al estado interno)
# ---------------------------------------------------------------------------

def sha256_padding(msg_len_bytes: int) -> bytes:
    """Calcula el padding MD para un mensaje de `msg_len_bytes` bytes."""
    bit_len = msg_len_bytes * 8
    pad = b'\x80'
    pad += b'\x00' * ((55 - msg_len_bytes) % 64)
    pad += struct.pack('>Q', bit_len)
    return pad


def sha256_from_state(state_hex: str, data: bytes, fake_msg_len: int) -> bytes:
    """
    Continúa SHA-256 desde un estado intermedio conocido.
    fake_msg_len: longitud en bytes del mensaje que produjo el estado (incluyendo padding).
    """
    # Reconstruir estado desde el MAC
    digest_bytes = bytes.fromhex(state_hex)
    state = list(struct.unpack('>8I', digest_bytes))

    # Construir nuevo bloque con extensión
    new_msg = data
    # Calcular la longitud total fingida para el padding
    total_len = fake_msg_len + len(data)

    # Padding de la extensión
    bit_len = total_len * 8
    padded = new_msg + b'\x80'
    padded += b'\x00' * ((55 - total_len) % 64)
    padded += struct.pack('>Q', bit_len)

    # Procesar bloques de la extensión
    new_state = state[:]
    for i in range(0, len(padded), 64):
        new_state = SHA256._compress(new_state, padded[i:i+64])

    return struct.pack('>8I', *new_state)


# ---------------------------------------------------------------------------
# Simulación del ataque
# ---------------------------------------------------------------------------

secret  = b'S3cr3t_K3y_!'          # 12 bytes, desconocido para el atacante
message = b'user=alice&action=read'

# Servidor: MAC = SHA-256(secret ‖ message)
legitimate_mac = hashlib.sha256(secret + message).hexdigest()
print(f'MAC legítimo     : {legitimate_mac}')
print(f'len(secret)      : {len(secret)} bytes  (supuesto conocido por atacante)')

# ---------------------------------------------------------------------------
# Atacante: extiende sin conocer secret
extension = b'&action=delete&admin=true'

# El atacante sabe: MAC, len(secret)=12, message
# Longitud del mensaje procesado (secret + message + padding):
inner_len  = len(secret) + len(message)
inner_pad  = sha256_padding(inner_len)
fake_len   = inner_len + len(inner_pad)   # longitud efectiva hasta el estado

forged_mac = sha256_from_state(legitimate_mac, extension, fake_len)
print(f'\nMAC forjado      : {forged_mac.hex()}')

# El servidor recibe: message ‖ padding ‖ extension y forged_mac
# Lo que el servidor computa "en realidad": H(secret ‖ message ‖ padding ‖ extension)
replicated_input = secret + message + inner_pad + extension
server_check     = hashlib.sha256(replicated_input).hexdigest()
print(f'Verificación srv : {server_check}')
print(f'\nAtaque exitoso   : {forged_mac.hex() == server_check}')
print(f'\nMensaje forjado enviado al servidor:')
print(f'  {(message + inner_pad + extension)!r}')


<a id='8-3'></a>
### 8.3 Mitigaciones

#### 1. HMAC (RFC 2104)

```
HMAC(K, m) = H( (K ⊕ opad) ‖ H( (K ⊕ ipad) ‖ m ) )
             └────────────────────────────────────────┘
             Doble hash rompe la extensión MD
```

HMAC-SHA256 **no es vulnerable** al length-extension attack.

#### 2. SHA-3 / Keccak

La construcción esponja no expone el estado completo en la salida; la capacidad `c` permanece oculta.

#### 3. BLAKE2 / BLAKE3

Usan contadores de bytes procesados dentro del estado; el estado no se puede reutilizar externamente.

#### 4. SHA-512/256 o SHA-384

Truncan la salida a menos de 512/384 bits, ocultando parte del estado.


In [ ]:
import hmac as hmac_std

# Comparación: MAC vulnerable vs HMAC seguro
secret  = b'S3cr3t_K3y_!'
message = b'user=alice&action=read'

# MAC vulnerable (NO usar en producción)
vuln_mac = hashlib.sha256(secret + message).hexdigest()

# HMAC-SHA256 (correcto)
secure_mac = hmac_std.new(secret, message, hashlib.sha256).hexdigest()

# Intento de extensión en HMAC → falla
# HMAC(k, m ‖ ext) ≠ computado desde HMAC(k, m)
extension  = b'&action=delete&admin=true'
inner_pad  = sha256_padding(len(secret) + len(message))
fake_len   = len(secret) + len(message) + len(inner_pad)
forged     = sha256_from_state(vuln_mac, extension, fake_len)

replicated = secret + message + inner_pad + extension
hmac_ext   = hmac_std.new(secret, message + inner_pad + extension, hashlib.sha256).hexdigest()

print('=== MAC vulnerables (SHA-256 concatenado) ===')
print(f'MAC original  : {vuln_mac}')
print(f'MAC forjado   : {forged.hex()}')
print(f'Ataque exitoso: {forged.hex() == hashlib.sha256(replicated).hexdigest()}')

print()
print('=== HMAC-SHA256 ===')
print(f'HMAC original         : {secure_mac}')
print(f'HMAC(k, msg‖pad‖ext)  : {hmac_ext}')
print(f'¿Forjado==HMAC(ext)?  : {forged.hex() == hmac_ext}   ← Ataque FALLA')

print()
print('=== SHA-3 (no vulnerable) ===')
sha3_mac   = hashlib.sha3_256(secret + message).hexdigest()
print(f'SHA3-256 MAC : {sha3_mac}')
print('SHA-3 no expone estado completo → extensión no factible sin clave')


---
<a id='9'></a>
## 9. Comparativa de rendimiento

Comparamos el rendimiento de las principales funciones hash procesando distintos tamaños de datos.


In [ ]:
import time, hashlib

algorithms = [
    ('MD5 (inseguro)',    'md5'),
    ('SHA-1 (inseguro)', 'sha1'),
    ('SHA-224',          'sha224'),
    ('SHA-256',          'sha256'),
    ('SHA-384',          'sha384'),
    ('SHA-512',          'sha512'),
    ('SHA3-256',         'sha3_256'),
    ('SHA3-512',         'sha3_512'),
    ('BLAKE2b-256',      'blake2b'),
    ('BLAKE2s-256',      'blake2s'),
]

data_sizes = [1_000, 10_000, 100_000, 1_000_000]  # bytes
n_reps     = 20
results    = {}

for name, alg in algorithms:
    times = []
    for sz in data_sizes:
        data  = secrets.token_bytes(sz)
        start = time.perf_counter()
        for _ in range(n_reps):
            hashlib.new(alg, data).digest()
        elapsed = (time.perf_counter() - start) / n_reps
        throughput_mb = sz / elapsed / 1e6   # MB/s
        times.append(throughput_mb)
    results[name] = times

# Tabla de resultados
print(f'{"Algoritmo":<18}', '  '.join(f'{sz//1000:>8}kB' for sz in data_sizes))
print('-' * 70)
for name, times in results.items():
    row = '  '.join(f'{t:>8.1f}' for t in times)
    print(f'{name:<18} {row}  MB/s')


In [ ]:
# Gráfico de throughput para bloques de 100 kB
idx_100k = data_sizes.index(100_000)
throughputs = {name: times[idx_100k] for name, times in results.items()}

sorted_items = sorted(throughputs.items(), key=lambda x: x[1])
names  = [k for k, _ in sorted_items]
values = [v for _, v in sorted_items]
colors_bars = ['#E74C3C' if 'inseguro' in n else
               '#F39C12' if 'SHA3' in n else
               '#27AE60' if 'BLAKE' in n else
               '#2980B9'
               for n in names]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(names, values, color=colors_bars, alpha=0.85)
ax.bar_label(bars, fmt='{:.0f} MB/s', padding=5, fontsize=9)
ax.set_xlabel('Throughput (MB/s) — bloques de 100 kB')
ax.set_title('Rendimiento comparativo de funciones hash criptográficas (Python hashlib)')
ax.axvline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()


---
<a id='10'></a>
## 10. Ejercicios propuestos

### Ejercicio 1 — SHA-256 manual
Extender la implementación `SHA256` de la sección 3.3 para soportar SHA-224 (mismo algoritmo, IV diferente, output truncado a 224 bits). Verificar contra los vectores de prueba del FIPS 180-4.

### Ejercicio 2 — Merkle Tree con BLAKE2b
Modificar la clase `MerkleTree` para usar BLAKE2b en lugar de SHA-256 doble. Construir un árbol con 13 hojas (número impar) y verificar que la prueba de inclusión sigue funcionando correctamente.

### Ejercicio 3 — Birthday attack en 24 bits
Adaptar la función `birthday_attack` para operar con digests de 24 bits. Medir experimentalmente el número promedio de evaluaciones necesarias para encontrar una colisión y comparar con la predicción teórica `1.177 · 2^12`.

### Ejercicio 4 — HMAC vs MAC naïf
Diseñar una prueba que demuestre que HMAC-SHA256 es resistente al length-extension attack:
1. Calcular `HMAC(key, message)`.
2. Intentar forjar `HMAC(key, message ‖ padding ‖ extension)` usando solo el valor del primer HMAC.
3. Verificar que el intento falla.

### Ejercicio 5 — Verificación de integridad de archivos
Implementar una función `verify_download(url, expected_sha256)` que:
1. Descargue un archivo en modo streaming.
2. Calcule el SHA-256 progresivamente.
3. Compare con el hash esperado y levante `ValueError` si difieren.

### Ejercicio 6 — Árbol de Merkle en Bitcoin
Descargar los datos del bloque Bitcoin #800000 usando la API de `blockstream.info` y:
1. Extraer todas las txids.
2. Reconstruir el Merkle root.
3. Comparar con el Merkle root en el encabezado del bloque.

> 💡 **Hint:** en Bitcoin, el hash de cada transacción usa doble SHA-256 (`SHA256d`), y los txids se expresan en *little-endian* antes de hashearlos.

### Ejercicio 7 — Análisis estadístico de bits
Generar 10 000 hashes SHA-256 de mensajes aleatorios y calcular:
- La frecuencia de `0` y `1` por posición de bit.
- El coeficiente de correlación entre bits adyacentes.
- La entropía estimada de la secuencia de digests.

Comparar los resultados con los de SHA3-256 y BLAKE2b.


---
<a id='11'></a>
## 11. Referencias

### Estándares y especificaciones

1. **FIPS PUB 180-4** — *Secure Hash Standard (SHS)*. NIST, 2015.  
   https://doi.org/10.6028/NIST.FIPS.180-4

2. **FIPS PUB 202** — *SHA-3 Standard: Permutation-Based Hash and Extendable-Output Functions*. NIST, 2015.  
   https://doi.org/10.6028/NIST.FIPS.202

3. **RFC 6234** — *US Secure Hash Algorithms (SHA and SHA-based HMAC and HKDF)*. Eastlake, Hansen. IETF, 2011.  
   https://www.rfc-editor.org/rfc/rfc6234

4. **RFC 2104** — *HMAC: Keyed-Hashing for Message Authentication*. Krawczyk, Bellare, Canetti. IETF, 1997.  
   https://www.rfc-editor.org/rfc/rfc2104

### Artículos académicos

5. **Bertoni, G., Daemen, J., Peeters, M., Van Assche, G.** — *Keccak*. EUROCRYPT 2013.  
   https://keccak.team/keccak.html

6. **Aumasson, J.-P., Neves, S., Wilcox-O'Hearn, Z., Winnerlein, C.** — *BLAKE2: simpler, smaller, fast as MD5*. ACNS 2013.  
   https://eprint.iacr.org/2013/322

7. **O'Connor, J., et al.** — *BLAKE3 — One Function, Fast Everywhere*. 2020.  
   https://github.com/BLAKE3-team/BLAKE3-specs/blob/master/blake3.pdf

8. **Kelsey, J., Schneier, B.** — *Second Preimages on n-bit Hash Functions for Much Less than 2ⁿ Work*. EUROCRYPT 2005.  
   https://eprint.iacr.org/2004/304

9. **Stevens, M., Bursztein, E., Karpman, P., Albertini, A., Markov, Y.** — *The First Collision for Full SHA-1*. CRYPTO 2017.  
   https://shattered.io

10. **Merkle, R. C.** — *Protocols for Public Key Cryptosystems*. IEEE S&P 1980.

### Libros de texto

11. **Boneh, D., Shoup, V.** — *A Graduate Course in Applied Cryptography*. Versión 0.6, 2023.  
    https://toc.cryptobook.us/

12. **Ferguson, N., Schneier, B., Kohno, T.** — *Cryptography Engineering*. Wiley, 2010. Capítulos 5–6.

13. **Paar, C., Pelzl, J.** — *Understanding Cryptography*. Springer, 2010. Capítulo 11.

### Recursos online

14. **Keccak Team** — referencia oficial de SHA-3/Keccak.  
    https://keccak.team

15. **BLAKE3 Team** — especificación y benchmarks.  
    https://blake3.io

16. **hashlib — Python 3 Documentation.**  
    https://docs.python.org/3/library/hashlib.html

17. **NIST Hash Functions** — página oficial con vectores de prueba.  
    https://csrc.nist.gov/projects/hash-functions

18. **"Hash Length Extension Attacks"** — blog post detallado por Valsorda.  
    https://words.filippo.io/the-ecb-penguin/ (contexto) y recursos en SkullSecurity.
